In [2]:
import pandas as pd
import nltk
import jieba
import MeCab
langs = ['Arabic', 'Chinese', "French", 'Japanese', "Russian"]
dataframes = {}
for lang in langs:
    dataframes[lang] = pd.read_csv(f"/home/jiaruil5/multilingual/multilingual-model-card/multilingualmc/dictionary_collection/mturk/analysis/annotation_final/{lang}.csv")
    
# Function to count words in a term based on language
def count_words(text, lang):
    if lang in ['English', 'French', 'Arabic', 'Russian']:
        return len(nltk.word_tokenize(text))
    elif lang == 'Chinese':
        return len([i for i in jieba.cut(text, cut_all=False)])
    elif lang == 'Japanese':
        mecab = MeCab.Tagger('-Owakati')
        return len(mecab.parse(text.strip()).split())

# Function to compute statistics
def compute_stats(dataframes):
    stats = {
        "# Terms": [],
        "Unique En Words": [],
        "Unique Tgt Words": [],
        "En Words/Term": [],
        "Tgt Words/Term": [],
        "En Chars/Term": [],
        "Tgt Chars/Term": []
    }
    
    for lang, df in dataframes.items():
        en_terms = df['English'].tolist()
        tgt_terms = df[lang].tolist()

        # Compute number of terms
        num_terms = len(en_terms)
        stats["# Terms"].append(num_terms)

        # Compute unique words in English and target language
        en_words = [word for term in en_terms for word in nltk.word_tokenize(term)]
        tgt_words = [word for term in tgt_terms for word in nltk.word_tokenize(term)] if lang != 'Chinese' and lang != 'Japanese' else \
                    [word for term in tgt_terms for word in jieba.cut(term, cut_all=False)] if lang == 'Chinese' else \
                    [word for term in tgt_terms for word in MeCab.Tagger('-Owakati').parse(term.strip()).split()]
        stats["Unique En Words"].append(len(set(en_words)))
        stats["Unique Tgt Words"].append(len(set(tgt_words)))

        # Compute average words per term
        en_words_per_term = sum(len(nltk.word_tokenize(term)) for term in en_terms) / num_terms
        tgt_words_per_term = sum(count_words(term, lang) for term in tgt_terms) / num_terms
        stats["En Words/Term"].append(en_words_per_term)
        stats["Tgt Words/Term"].append(tgt_words_per_term)

        # Compute average characters per term
        en_chars_per_term = sum(len(term) for term in en_terms) / num_terms
        tgt_chars_per_term = sum(len(term) for term in tgt_terms) / num_terms
        stats["En Chars/Term"].append(en_chars_per_term)
        stats["Tgt Chars/Term"].append(tgt_chars_per_term)

    return pd.DataFrame(stats, index=dataframes.keys())

# Generate the statistics
stats_df = compute_stats(dataframes)

Building prefix dict from the default dictionary ...
Loading model from cache /tmp/jieba.cache
Loading model cost 3.606 seconds.
Prefix dict has been built successfully.


In [8]:
print(stats_df.T.to_latex(float_format="%.2f"))

\begin{tabular}{lrrrrr}
\toprule
 & Arabic & Chinese & French & Japanese & Russian \\
\midrule
# Terms & 4844.00 & 6426.00 & 6527.00 & 4770.00 & 5167.00 \\
Unique En Words & 2470.00 & 3244.00 & 3470.00 & 2424.00 & 2615.00 \\
Unique Tgt Words & 3161.00 & 2838.00 & 4036.00 & 2050.00 & 4210.00 \\
En Words/Term & 2.02 & 2.05 & 2.07 & 2.02 & 2.01 \\
Tgt Words/Term & 2.36 & 2.26 & 2.68 & 2.53 & 2.16 \\
En Chars/Term & 16.99 & 17.26 & 17.44 & 16.96 & 16.94 \\
Tgt Chars/Term & 15.22 & 4.66 & 21.27 & 6.89 & 20.20 \\
\bottomrule
\end{tabular}



In [12]:
import numpy as np
def compute_stats_with_std(dataframes):
    stats = {
        "# Terms": [],
        "Unique En Words": [],
        "Unique Tgt Words": [],
        "En Words/Term": [],
        "Tgt Words/Term": [],
        "En Chars/Term": [],
        "Tgt Chars/Term": []
    }
    
    for lang, df in dataframes.items():
        en_terms = df['English'].tolist()
        tgt_terms = df[lang].tolist()

        # Compute number of terms
        num_terms = len(en_terms)
        stats["# Terms"].append(num_terms)

        # Compute unique words in English and target language
        en_words = [word for term in en_terms for word in nltk.word_tokenize(term)]
        tgt_words = [word for term in tgt_terms for word in nltk.word_tokenize(term)] if lang != 'Chinese' and lang != 'Japanese' else \
                    [word for term in tgt_terms for word in jieba.cut(term, cut_all=False)] if lang == 'Chinese' else \
                    [word for term in tgt_terms for word in MeCab.Tagger('-Owakati').parse(term.strip()).split()]
        stats["Unique En Words"].append(len(set(en_words)))
        stats["Unique Tgt Words"].append(len(set(tgt_words)))

        # Compute mean and std for words per term
        en_words_per_term = [len(nltk.word_tokenize(term)) for term in en_terms]
        tgt_words_per_term = [count_words(term, lang) for term in tgt_terms]
        stats["En Words/Term"].append(f"{np.mean(en_words_per_term):.2f} \pm {np.std(en_words_per_term):.2f}")
        stats["Tgt Words/Term"].append(f"{np.mean(tgt_words_per_term):.2f} \pm {np.std(tgt_words_per_term):.2f}")

        # Compute mean and std for characters per term
        en_chars_per_term = [len(term) for term in en_terms]
        tgt_chars_per_term = [len(term) for term in tgt_terms]
        stats["En Chars/Term"].append(f"{np.mean(en_chars_per_term):.2f} \pm {np.std(en_chars_per_term):.2f}")
        stats["Tgt Chars/Term"].append(f"{np.mean(tgt_chars_per_term):.2f} \pm {np.std(tgt_chars_per_term):.2f}")

    return pd.DataFrame(stats, index=dataframes.keys())

# Generate the statistics with mean and std
stats_df_with_std = compute_stats_with_std(dataframes)

In [13]:
print(stats_df_with_std.T.to_latex())

\begin{tabular}{llllll}
\toprule
 & Arabic & Chinese & French & Japanese & Russian \\
\midrule
# Terms & 4844 & 6426 & 6527 & 4770 & 5167 \\
Unique En Words & 2470 & 3244 & 3470 & 2424 & 2615 \\
Unique Tgt Words & 3161 & 2838 & 4036 & 2050 & 4210 \\
En Words/Term & 2.02 \pm 0.59 & 2.05 \pm 0.68 & 2.07 \pm 0.67 & 2.02 \pm 0.58 & 2.01 \pm 0.59 \\
Tgt Words/Term & 2.36 \pm 0.83 & 2.26 \pm 0.90 & 2.68 \pm 1.19 & 2.53 \pm 0.98 & 2.16 \pm 0.80 \\
En Chars/Term & 16.99 \pm 5.97 & 17.26 \pm 6.60 & 17.44 \pm 6.57 & 16.96 \pm 5.91 & 16.94 \pm 5.90 \\
Tgt Chars/Term & 15.22 \pm 5.66 & 4.66 \pm 1.96 & 21.27 \pm 8.49 & 6.89 \pm 3.16 & 20.20 \pm 7.83 \\
\bottomrule
\end{tabular}



In [16]:
import json
data = json.load(open("/home/jiaruil5/multilingual/multilingual-model-card/multilingualmc/dataset/info.json", 'r'))
len(data['dev']['paper']) + len(data['test']['paper'])

879